In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import average_precision_score, mutual_info_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import ParameterSampler

import produce_features as pf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})

DATA = "data/synthetic_produce_prices_1990_2025.csv"
HORIZONS = list(pf.HORIZONS)

In [ ]:
raw = pd.read_csv(DATA)
print(raw.shape)
display(raw.head(3))
display(raw.dtypes.to_frame("dtype").T)

In [ ]:
df = raw.copy()
df["date"] = pd.to_datetime(df["date"])

checks = pd.Series({
    "missing cells": int(df.isna().sum().sum()),
    "duplicate rows": int(df.duplicated().sum()),
    "duplicate (date, store, product)": int(df.duplicated(["date", "store_id", "product"]).sum()),
    "non-positive prices": int((df["price_usd_per_lb"] <= 0).sum()),
    "availability outside 0-100": int((~df["availability_pct"].between(0, 100)).sum()),
    "dates not on the 1st of a month": int((df["date"].dt.day != 1).sum()),
    "shortage_flag disagrees with availability < 82": int(((df["availability_pct"] < 82).astype(int) != df["shortage_flag"]).sum()),
    "frost/heat days outside 0-31": int((~df["frost_days"].between(0, 31) | ~df["heatwave_days"].between(0, 31)).sum()),
})
display(checks.to_frame("count"))
assert (checks == 0).all()

In [ ]:
display(df.describe().T[["min", "25%", "50%", "75%", "max"]].round(2))
for c in ["store_id", "region", "store_tier", "product", "unit"]:
    print(c, "->", sorted(df[c].unique()))

In [ ]:
# Months with no price report: each store-product series should have one row per month.
months = pd.date_range(df["date"].min(), df["date"].max(), freq="MS")
per_series = df.groupby(["store_id", "product"]).size()
print(f"{len(months)} months, {len(per_series)} series; rows per series: min {per_series.min()}, max {per_series.max()}")
print(f"overall share of months with no report: {1 - len(df) / (len(months) * len(per_series)):.1%}")

# Longest run of consecutive missing months in any series
longest = 0
for _, g in df.groupby(["store_id", "product"]):
    idx = months.get_indexer(g["date"])
    gaps = np.diff(np.concatenate([[-1], idx, [len(months)]])) - 1
    longest = max(longest, gaps.max())
print("longest run of consecutive missing months in any series:", longest)

In [ ]:
# Prices over time: nominal, and with inflation removed
df["real_price"] = np.exp(pf.real_log_price(df["price_usd_per_lb"], df["cpi_index"]))
nominal = df.groupby(["year", "product"])["price_usd_per_lb"].mean().unstack()
real = df.groupby(["year", "product"])["real_price"].mean().unstack()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
nominal.plot(ax=ax[0], legend=False)
ax[0].set_title("Average nominal price (USD/lb)")
(real / real.iloc[0]).plot(ax=ax[1])
ax[1].set_title("Inflation-adjusted price, 1990 = 1")
ax[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonality of availability by product
seasonal = df.groupby(["product", "month"])["availability_pct"].mean().unstack()
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(seasonal.values, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(seasonal)))
ax.set_yticklabels(seasonal.index)
ax.set_xticks(range(12))
ax.set_xticklabels(range(1, 13))
ax.set_xlabel("Month")
ax.set_title("Mean availability (%) by product and month")
ax.grid(False)
plt.colorbar(im, label="availability (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Where do shortages happen?
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df.groupby("product")["shortage_flag"].mean().sort_values().plot.barh(ax=ax[0], title="Shortage rate by product")
df.groupby("store_id")["shortage_flag"].mean().plot.bar(ax=ax[1], title="Shortage rate by store", rot=0)
df.groupby("year")["shortage_flag"].mean().plot(ax=ax[2], title="Shortage rate by year")
plt.tight_layout()
plt.show()
print(f"overall shortage rate: {df['shortage_flag'].mean():.1%}")

In [ ]:
# Weather effect on availability, after removing the usual seasonal pattern
d = df.copy()
d["avail_resid"] = d["availability_pct"] - d.groupby(["store_id", "product", "month"])["availability_pct"].transform("mean")

d["drought_bin"] = pd.qcut(d["drought_index"], 8)
sel = ["tomatoes", "avocados", "lettuce", "bananas", "onions"]
tab = d[d["product"].isin(sel)].groupby(["drought_bin", "product"], observed=True)["avail_resid"].mean().unstack()
tab.index = [round(iv.mid, 2) for iv in tab.index]
tab.plot(marker="o")
plt.xlabel("Drought index (negative = dry, positive = wet)")
plt.ylabel("Availability vs. usual for that month (points)")
plt.title("Dry spells cut availability, but not equally for every product")
plt.show()

wcols = ["drought_index", "temp_anomaly_c", "frost_days", "heatwave_days", "precip_mm"]
corr = pd.DataFrame({p: g[wcols].corrwith(g["avail_resid"]) for p, g in d.groupby("product")}).T
display(corr.round(2))

In [ ]:
# How long does weather take to show up in availability?
# Correlate availability with the drought index k months EARLIER (matched by date).
res = {}
for prod in ["tomatoes", "avocados", "lettuce"]:
    dp = d[d["product"] == prod]
    vals = []
    for k in range(0, 7):
        lagged = dp[["store_id", "date", "drought_index"]].copy()
        lagged["date"] = lagged["date"] + pd.DateOffset(months=k)
        m = dp.merge(lagged, on=["store_id", "date"], suffixes=("", "_lag"))
        vals.append(m["avail_resid"].corr(m["drought_index_lag"]))
    res[prod] = vals
pd.DataFrame(res, index=range(0, 7)).plot(marker="o")
plt.xlabel("Months between the drought index and availability (k)")
plt.ylabel("Correlation with availability")
plt.title("Weather effect fades over a few months")
plt.show()

In [ ]:
panel = pf.build_panel(df)
print(panel.shape, "rows (24 series x 432 months)")
cols = ["t", "avail", "avail_lag1", "avail_lag2", "avail_lag12", "real_logprice", "drought_index", "drought_index_3m", "y_h1", "y_h3"]
display(panel[(panel.store_id == "S01") & (panel["product"] == "strawberries")][cols].iloc[24:30].round(2))

In [ ]:
frames = {h: pf.horizon_frame(panel, h) for h in HORIZONS}

rows = []
for h in HORIZONS:
    tr, va, te = pf.split_by_target_year(frames[h])
    rows.append({"horizon": h, "train": len(tr), "val": len(va), "test": len(te),
                 "shortage rate train": round(tr.y.mean(), 3),
                 "shortage rate val": round(va.y.mean(), 3), "shortage rate test": round(te.y.mean(), 3)})
display(pd.DataFrame(rows).set_index("horizon"))

In [ ]:
feats = pf.FEATURES
tr1, va1, te1 = pf.split_by_target_year(frames[1])
tr3, va3, te3 = pf.split_by_target_year(frames[3])

def mutual_info_table(tr):
    X = tr[pf.ALL_NUM].fillna(tr[pf.ALL_NUM].median())
    mi = pd.Series(mutual_info_classif(X, tr["y"], random_state=1), index=pf.ALL_NUM)
    for c in pf.CATEGORICAL:
        mi[c] = mutual_info_score(tr[c], tr["y"])
    return mi

mi = pd.DataFrame({"1 month ahead": mutual_info_table(tr1), "3 months ahead": mutual_info_table(tr3)})
mi = mi.sort_values("1 month ahead", ascending=True)
mi.plot.barh(figsize=(9, 8), title="Mutual information with next shortage (training years)")
plt.tight_layout()
plt.show()

In [ ]:
def permutation_by_group(model, X, y, groups, n_repeats=5, seed=1):
    """Drop in PR-AUC when a group of columns is shuffled together."""
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, model.predict_proba(X)[:, 1])
    out = {}
    for name, cols in groups.items():
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].values[rng.permutation(len(X))]
            drops.append(base - average_precision_score(y, model.predict_proba(Xp)[:, 1]))
        out[name] = float(np.mean(drops))
    return pd.Series(out).sort_values(ascending=False), base

single = {c: [c] for c in feats}
groups = {
    "availability (now and past)": ["avail", "avail_lag1", "avail_lag2", "avail_lag3", "avail_lag12"],
    "inflation-adjusted price": ["real_logprice", "real_logprice_lag1", "real_logprice_lag12"],
    "weather and ENSO": pf.WEATHER_NUM,
    "month being predicted": ["target_month_sin", "target_month_cos"],
    "fuel cost": ["fuel_cost_index"],
    "store and product": pf.CATEGORICAL,
}

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
for a, (h, tr, va) in zip(ax, [(1, tr1, va1), (3, tr3, va3)]):
    m = pf.make_model("boosting", pf.ALL_NUM).fit(tr[feats], tr["y"])
    g, base = permutation_by_group(m, va[feats], va["y"], groups)
    g.iloc[::-1].plot.barh(ax=a, title=f"{h} month(s) ahead (validation PR-AUC {base:.2f})")
    a.set_xlabel("drop in PR-AUC when shuffled")
    print(f"{h} month(s) ahead:"); print(g.round(3).to_string()); print()
plt.tight_layout()
plt.show()

In [ ]:
# The same, one feature at a time (1 and 3 months ahead)
top = {}
for h, tr, va in [(1, tr1, va1), (3, tr3, va3)]:
    m = pf.make_model("boosting", pf.ALL_NUM).fit(tr[feats], tr["y"])
    s, _ = permutation_by_group(m, va[feats], va["y"], single)
    top[f"{h} month(s) ahead"] = s
top = pd.DataFrame(top).sort_values("1 month(s) ahead", ascending=False)
display(top.head(12).round(4))

In [ ]:
def val_pr_auc(kind, num_cols, params=None):
    scores = []
    for h in HORIZONS:
        tr, va, _ = pf.split_by_target_year(frames[h])
        cols = num_cols + pf.CATEGORICAL
        m = pf.make_model(kind, num_cols, params).fit(tr[cols], tr["y"])
        scores.append(average_precision_score(va["y"], m.predict_proba(va[cols])[:, 1]))
    return scores

candidates = {
    "logistic regression (no weather)": ("logistic", pf.BASE_NUM),
    "logistic regression (+ weather)": ("logistic", pf.ALL_NUM),
    "random forest (no weather)": ("forest", pf.BASE_NUM),
    "random forest (+ weather)": ("forest", pf.ALL_NUM),
    "gradient boosting (no weather)": ("boosting", pf.BASE_NUM),
    "gradient boosting (+ weather)": ("boosting", pf.ALL_NUM),
}

rows = {}
for name, (kind, cols) in candidates.items():
    rows[name] = val_pr_auc(kind, cols)

base_rows = {"baseline: never shortage": [], "baseline: persistence": [], "baseline: same month last year": []}
for h in HORIZONS:
    _, va, _ = pf.split_by_target_year(frames[h])
    base_rows["baseline: never shortage"].append(average_precision_score(va["y"], np.zeros(len(va))))
    base_rows["baseline: persistence"].append(average_precision_score(va["y"], va["persist"]))
    base_rows["baseline: same month last year"].append(average_precision_score(va["y"], va["seasonal"]))
rows.update(base_rows)

sel = pd.DataFrame(rows, index=[f"{h} month(s)" for h in HORIZONS]).T
sel["mean"] = sel.mean(axis=1)
sel = sel.sort_values("mean", ascending=False)
display(sel.round(3))

sel["mean"].iloc[::-1].plot.barh(figsize=(9, 4.5), title="Validation PR-AUC, mean over 1-3 months ahead")
plt.tight_layout()
plt.show()

In [ ]:
# Does adding weather help, for each family?
for fam in ["logistic regression", "random forest", "gradient boosting"]:
    gain = sel.loc[f"{fam} (+ weather)", sel.columns[:3]] - sel.loc[f"{fam} (no weather)", sel.columns[:3]]
    print(f"{fam:20s} PR-AUC gain from weather:", gain.round(3).to_dict())

In [ ]:
FOLDS = [(2006, 2010), (2011, 2015), (2016, 2020)]
feats = pf.FEATURES

def cv_score(kind, params=None, num_cols=pf.ALL_NUM):
    """Mean PR-AUC per fold (averaged over horizons)."""
    cols = num_cols + pf.CATEGORICAL
    fold_scores = []
    for lo, hi in FOLDS:
        per_h = []
        for h in HORIZONS:
            d = frames[h]
            year = d["target_date"].dt.year
            tr, va = d[year < lo], d[(year >= lo) & (year <= hi)]
            m = pf.make_model(kind, num_cols, params).fit(tr[cols], tr["y"])
            per_h.append(average_precision_score(va["y"], m.predict_proba(va[cols])[:, 1]))
        fold_scores.append(np.mean(per_h))
    return fold_scores

space = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.12],
    "max_depth": [2, 3, 4, 5, 6],
    "max_iter": [100, 200, 300, 500],
    "min_samples_leaf": [10, 20, 40, 80],
    "l2_regularization": [0.0, 0.1, 1.0, 10.0],
    "max_leaf_nodes": [8, 15, 31],
}
configs = [dict(pf.DEFAULT_HGB_PARAMS)] + [dict(p) for p in ParameterSampler(space, n_iter=24, random_state=1)]

results = []
for i, cfg in enumerate(configs):
    scores = cv_score("boosting", cfg)
    results.append({"config": "default" if i == 0 else f"random {i}", **cfg,
                    "cv_mean": np.mean(scores), "cv_std": np.std(scores)})
tuning = pd.DataFrame(results).sort_values("cv_mean", ascending=False).reset_index(drop=True)
display(tuning.head(8).round(4))
print("default settings rank:", int(tuning.index[tuning["config"] == "default"][0]) + 1, "of", len(tuning))

In [ ]:
# Logistic regression and random forest on the same folds, for a fair comparison with tuned boosting
ref = {}
for C in [0.01, 0.1, 1, 10, 100]:
    ref[f"logistic C={C} (+ weather)"] = cv_score("logistic", {"C": C})
ref["random forest (defaults, + weather)"] = cv_score("forest")
ref["boosting (defaults, + weather)"] = cv_score("boosting", dict(pf.DEFAULT_HGB_PARAMS))
best_row = tuning.iloc[0]
best_params = {k: (int(best_row[k]) if k in ("max_iter", "max_depth", "min_samples_leaf", "max_leaf_nodes", "random_state")
                   else float(best_row[k])) for k in ["learning_rate", "max_depth", "max_iter", "min_samples_leaf",
                                                    "l2_regularization", "max_leaf_nodes", "random_state"]
               if k in best_row.index and pd.notna(best_row[k])}
ref["boosting (tuned, + weather)"] = cv_score("boosting", best_params)

cmp_ = pd.DataFrame({k: {"cv_mean": np.mean(v), "cv_std": np.std(v)} for k, v in ref.items()}).T
display(cmp_.sort_values("cv_mean", ascending=False).round(4))
print("best parameters:", best_params)

In [ ]:
# Save the chosen hyperparameters; train.py reads this file
with open("model_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
print(open("model_params.json").read())

In [ ]:
final, test_rows, curves = {}, [], {}
for h in HORIZONS:
    tr, va, te = pf.split_by_target_year(frames[h])
    probe = pf.make_model("boosting", pf.ALL_NUM, best_params).fit(tr[feats], tr["y"])
    thr = pf.best_f1_threshold(va["y"], probe.predict_proba(va[feats])[:, 1])

    trva = pd.concat([tr, va])
    model = pf.make_model("boosting", pf.ALL_NUM, best_params).fit(trva[feats], trva["y"])
    p = model.predict_proba(te[feats])[:, 1]
    prec, rec = pf.precision_recall_at(te["y"], (p >= thr).astype(int))
    final[h] = dict(model=model, threshold=thr, p=p, te=te)
    curves[h] = precision_recall_curve(te["y"], p)

    test_rows.append({"horizon": h, "model": "gradient boosting (tuned)",
                      "PR-AUC": average_precision_score(te["y"], p), "ROC-AUC": roc_auc_score(te["y"], p),
                      "precision": prec, "recall": rec, "threshold": thr})
    for name, s in [("baseline: never shortage", np.zeros(len(te))),
                    ("baseline: persistence", te["persist"].values),
                    ("baseline: same month last year", te["seasonal"].values)]:
        pr_, rc_ = pf.precision_recall_at(te["y"], (s >= 0.5).astype(int))
        test_rows.append({"horizon": h, "model": name, "PR-AUC": average_precision_score(te["y"], s),
                          "ROC-AUC": roc_auc_score(te["y"], s) if s.std() > 0 else 0.5,
                          "precision": pr_, "recall": rc_, "threshold": np.nan})

test_table = pd.DataFrame(test_rows).set_index(["horizon", "model"])
display(test_table.round(3))
print("shortage rate in the test years:", round(final[1]["te"]["y"].mean(), 3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for a, h in zip(ax, HORIZONS):
    p_, r_, _ = curves[h]
    a.plot(r_, p_, label="model")
    a.axhline(final[h]["te"]["y"].mean(), color="gray", linestyle="--", label="no skill")
    a.scatter([test_table.loc[(h, "gradient boosting (tuned)"), "recall"]],
              [test_table.loc[(h, "gradient boosting (tuned)"), "precision"]], color="red", zorder=3, label="chosen threshold")
    a.set_title(f"{h} month(s) ahead")
    a.set_xlabel("recall")
ax[0].set_ylabel("precision")
ax[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# One month ahead: confusion matrix, calibration and results by product
te = final[1]["te"].copy()
te["p"] = final[1]["p"]
te["warn"] = (te["p"] >= final[1]["threshold"]).astype(int)
cm = pd.crosstab(te["y"].map({0: "no shortage", 1: "shortage"}), te["warn"].map({0: "no warning", 1: "warning"}))
display(cm)

bins = pd.cut(te["p"], [0, 0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0], include_lowest=True)
calib = te.groupby(bins, observed=True).agg(rows=("y", "size"), mean_predicted=("p", "mean"), actual_rate=("y", "mean"))
display(calib.round(3))

by_prod = te.groupby("product").apply(lambda g: pd.Series({
    "rows": len(g), "shortages": int(g["y"].sum()),
    "recall": g.loc[g["y"] == 1, "warn"].mean() if g["y"].sum() else np.nan,
    "PR-AUC": average_precision_score(g["y"], g["p"]) if g["y"].sum() else np.nan}))
display(by_prod.round(2))